# Arbitrary Value Imputation

Instead of filling missing values with something "realistic" (like the mean), **arbitrary value imputation** fills them with a value chosen specifically to be **obviously outside** the normal range (e.g. `99`, `-1`) — the point is to make "this value was missing" visible to the model as a distinct signal, rather than pretending it was a typical observation. Works well with tree-based models, which can easily learn "this weird value means missing"; poorly with linear models, which have no such flexibility.

## 1. Setup and Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [3]:
df = pd.read_csv('titanic_toy.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'titanic_toy.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().mean()

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

## 2. Imputing With Out-of-Range Values

`Age` never naturally reaches 99, and `Fare` never naturally reaches 999 — anything a model sees at that value can only mean "this was missing." `-1` works the same way at the other end (below any real age/fare).

In [ ]:
X_train['Age_99'] = X_train['Age'].fillna(99)
X_train['Age_minus1'] = X_train['Age'].fillna(-1)

X_train['Fare_999'] = X_train['Fare'].fillna(999)
X_train['Fare_minus1'] = X_train['Fare'].fillna(-1)

### Checking the Effect on Variance

Any single-value imputation (arbitrary value included) artificially **changes variance** — an arbitrary far-out value inflates it a lot (since the imputed rows sit far from everything else), unlike mean/median imputation, which shrinks variance instead.

In [ ]:
print('Original Age variable variance: ', X_train['Age'].var())
print('Age Variance after 99 wala imputation: ', X_train['Age_99'].var())
print('Age Variance after -1 wala imputation: ', X_train['Age_minus1'].var())

print('Original Fare variable variance: ', X_train['Fare'].var())
print('Fare Variance after 999 wala imputation: ', X_train['Fare_999'].var())
print('Fare Variance after -1 wala imputation: ', X_train['Fare_minus1'].var())

### Visualizing the Distortion

Overlaying the original distribution against each imputed version shows exactly how visible the arbitrary value is — the imputed columns should show an obvious spike right at 99/-1 (or 999/-1) that has nothing to do with the real data's shape.

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original variable distribution
X_train['Age'].plot(kind='kde', ax=ax)

# variable imputed with the median
X_train['Age_99'].plot(kind='kde', ax=ax, color='red')

# variable imputed with the mean
X_train['Age_minus1'].plot(kind='kde', ax=ax, color='green')

# add legends
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='best')

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111)

# original variable distribution
X_train['Fare'].plot(kind='kde', ax=ax)

# variable imputed with the median
X_train['Fare_999'].plot(kind='kde', ax=ax, color='red')

# variable imputed with the mean
X_train['Fare_minus1'].plot(kind='kde', ax=ax, color='green')

# add legends
lines, labels = ax.get_legend_handles_labels()
ax.legend(lines, labels, loc='best')

### Checking Relationships Between Columns

`.cov()` and `.corr()` reveal whether the arbitrary values distorted relationships **between** columns, not just each column's own distribution — worth a quick look before deciding this is fine to use.

In [ ]:
X_train.cov()

In [ ]:
X_train.corr()

### Using Sklearn

## 3. Doing the Same Thing With `SimpleImputer`

`strategy='constant'` with `fill_value=...` is `SimpleImputer`'s version of arbitrary value imputation — same idea, but reusable across train/test consistently via `ColumnTransformer`, rather than repeating `.fillna()` calls by hand.

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
imputer1 = SimpleImputer(strategy='constant',fill_value=99)
imputer2 = SimpleImputer(strategy='constant',fill_value=999)

In [ ]:
trf = ColumnTransformer([
    ('imputer1',imputer1,['Age']),
    ('imputer2',imputer2,['Fare'])
],remainder='passthrough')

In [ ]:
trf.fit(X_train)

`.statistics_` here just confirms the fixed fill value each imputer was configured with — for `strategy='mean'`/`'median'`, this is where a *learned* value would show up instead.

In [ ]:
trf.named_transformers_['imputer1'].statistics_

In [ ]:
trf.named_transformers_['imputer2'].statistics_

In [ ]:
X_train = trf.transform(X_train)
X_test = trf.transform(X_test)

In [ ]:
X_train

## Summary

| Task | Method |
|---|---|
| Fill with an out-of-range value (manual) | `df['col'].fillna(99)` |
| Fill with an out-of-range value (sklearn, reusable) | `SimpleImputer(strategy='constant', fill_value=99)` |
| Check the distortion it introduces | `.var()` before/after, overlaid `.plot(kind='kde')` |
| Check relationships weren't badly distorted | `.cov()`, `.corr()` |

### When to reach for this vs. mean/median
- **Tree-based models** (decision trees, random forests, gradient boosting) → arbitrary value imputation works well; trees can isolate the weird value with a single split.
- **Linear models, KNN, anything distance/coefficient-based** → an arbitrary far-out value distorts the relationship badly; mean/median imputation (or a separate "was missing" indicator column) is usually safer.